# 第 04 章 · 工具系统：从模型意图到受控调用

模型提出 `write_file` 只是意图。执行前还需要回答：工具存在吗、参数符合合同吗、最终动作获准了吗、
这个调用是否已经受理过？把这些问题都放进文件 handler，既难审计，也容易在 Hook 改参数后继续沿用旧批准。

本章把 **注册 → 参数校验 → 不可变动作 → 授权 → 防重复受理 → 执行事实 → 观察** 拆开。
真实文件写入仅落在本章新目录。这里只研究分派机制，不把 Python 函数和内存 Permit 当成 OS 安全边界。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
tools_demo = load_example('04', 'toolbox')
from dataclasses import replace
import json
run_dir = fresh_run_dir('ch04')

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. 注册一个真的写入工具


    注册表保存参数合同、能力名和 handler。模型公开的工具说明只是入口描述，真正执行仍由宿主选择 handler。
    本例固定输出文件，模型只能提供正文，便于把注意力集中在注册与调用身份。
    修改 `content` 后观察磁盘内容和 handler 进入次数，而不是依赖一句“写入成功”。


In [3]:
registry = tools_demo.Registry()
output_file = run_dir / 'note.txt'
def write_note(text):
    output_file.write_text(text, encoding='utf-8')
    return {'bytes': output_file.stat().st_size}
registry.register('write_note', {'text': str}, 'workspace.write', write_note)
content = '需要修复 status 字段。'
action = registry.prepare('call-1', 'write_note', {'text': content})
permit = registry.authorize(action, {'workspace.write'})
observation = registry.dispatch(action, permit)
assert output_file.read_text() == content
assert observation['execution'] == 'succeeded'
assert registry.handler_entries == 1
{'action_digest': action.digest, 'observation': observation, 'file': str(output_file)}


{'action_digest': 'fbfba0057de57521255cb35982702f2411a3675b1b38b6231e5ff23300947021',
 'observation': {'digest': 'fbfba0057de57521255cb35982702f2411a3675b1b38b6231e5ff23300947021',
  'execution': 'succeeded',
  'observation': 'accepted',
  'result': {'bytes': 29}},
 'file': '<project>/runs/notebooks/ch04-20261113T124115Z-6cc0d405/note.txt'}

## 2. 拒绝应发生在 handler 之前


    未知工具、同名注册、参数类型错误和缺少能力许可是四种不同失败。对这些失败最有力的证据是 handler
    没有进入、文件没有改变，而不只是抛出了异常。本例只实现“精确键 + Python 类型”的小型校验器，
    并非完整 JSON Schema 实现。


In [4]:
before = output_file.read_bytes()
errors = [
    expect_error(ValueError, lambda: registry.register('write_note', {'text': str}, 'workspace.write', write_note), 'duplicate'),
    expect_error(ValueError, lambda: registry.prepare('c2', 'missing', {}), 'unknown_tool'),
    expect_error(ValueError, lambda: registry.prepare('c2', 'write_note', {'text': 3}), 'argument_type'),
    expect_error(PermissionError, lambda: registry.authorize(action, {'workspace.read'}), 'not_granted'),
]
assert registry.handler_entries == 1 and output_file.read_bytes() == before
errors


['duplicate_tool_name',
 'unknown_tool',
 'argument_type_mismatch',
 'capability_not_granted']

## 3. Hook 改写之后，旧批准失效


    批准应绑定最终动作。若批准正文 A 后才改成正文 B，名称仍是同一个工具也不能复用原许可。
    正确顺序是在 `prepare` 中改写与复验，然后为改写后的不可变动作签发 Permit。
    哈希帮助比较动作身份，并不自行提供权限。


In [5]:
rewrite_suffix = '\n先补失败测试。'
rewritten = registry.prepare('call-1', 'write_note', {'text': content},
                             rewrite=lambda args: {'text': args['text'] + rewrite_suffix})
mismatch = expect_error(PermissionError, lambda: registry.dispatch(rewritten, permit), 'permit_action_mismatch')
assert registry.handler_entries == 1
second = registry.prepare('call-2', 'write_note', {'text': content},
                          rewrite=lambda args: {'text': args['text'] + rewrite_suffix})
second_permit = registry.authorize(second, {'workspace.write'})
registry.dispatch(second, second_permit)
assert output_file.read_text() == content + rewrite_suffix
assert registry.handler_entries == 2
{'old_permit': mismatch, 'different_digest': action.digest != rewritten.digest}


{'old_permit': 'permit_action_mismatch', 'different_digest': True}

## 4. 重放同一调用，不重做副作用


    Dispatcher 在进入 handler 前登记调用身份。同一 call ID、同一动作的重复分派返回已保存记录；
    同一 call ID 换动作则拒绝。取消发生在受理前，也必须阻止 handler。
    这个账本只在内存中，进程重启后的恢复由第 10 章讨论，不能据此宣称跨崩溃 exactly-once。


In [6]:
first_receipt = registry.dispatch(second, second_permit)
assert registry.handler_entries == 2
assert first_receipt == registry.dispatch(second, second_permit)
changed = registry.prepare('call-2', 'write_note', {'text': '不同内容'})
changed_permit = registry.authorize(changed, {'workspace.write'})
reused = expect_error(ValueError, lambda: registry.dispatch(changed, changed_permit), 'reused_for_different')
cancelled_action = registry.prepare('call-3', 'write_note', {'text': '不能写入'})
cancelled_permit = registry.authorize(cancelled_action, {'workspace.write'})
cancelled = expect_error(ValueError, lambda: registry.dispatch(cancelled_action, cancelled_permit, cancelled=True), 'cancelled')
assert registry.handler_entries == 2
assert output_file.read_text() == content + rewrite_suffix
{'reused_id': reused, 'cancelled': cancelled}


{'reused_id': 'call_id_reused_for_different_action',
 'cancelled': 'cancelled_before_admission'}

## 5. 后置拒绝与丢失回执，不会撤回文件


    后置检查能拒绝把某段观察交给模型，却不能抹去已发生的写入。再构造“文件已写、handler 抛异常”的
    场景：分派器只知道结果不完整，必须保留 `unknown`，不能自动重试。
    人工核对当前文件是一个新的观察，不应倒填成过去已经确认的成功。


In [7]:
post_action = registry.prepare('call-4', 'write_note', {'text': '执行完成但观察被拒绝'})
post_permit = registry.authorize(post_action, {'workspace.write'})
rejected = registry.dispatch(post_action, post_permit, post_check=lambda record: False)
assert rejected['execution'] == 'succeeded' and rejected['observation'] == 'rejected'
assert output_file.read_text() == '执行完成但观察被拒绝'
def write_then_lose_reply(text):
    output_file.write_text(text, encoding='utf-8')
    raise ConnectionError('local injected reply loss')
registry.register('uncertain_write', {'text': str}, 'workspace.write', write_then_lose_reply)
uncertain_action = registry.prepare('call-5', 'uncertain_write', {'text': '现场已变化'})
uncertain_permit = registry.authorize(uncertain_action, {'workspace.write'})
uncertain = registry.dispatch(uncertain_action, uncertain_permit)
entries = registry.handler_entries
assert uncertain['execution'] == 'unknown'
assert output_file.read_text() == '现场已变化'
assert registry.dispatch(uncertain_action, uncertain_permit) == uncertain
assert registry.handler_entries == entries
{'post_check': rejected['observation'], 'lost_reply': uncertain, 'current_file': output_file.read_text()}


{'post_check': 'rejected',
 'lost_reply': {'digest': 'ed813a2a57ea5927367a885b0350c5e486e9d520491fb71f8b027a5ac8663e6d',
  'execution': 'unknown',
  'observation': 'accepted',
  'error': 'ConnectionError: local injected reply loss'},
 'current_file': '现场已变化'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [register_trusted_with_native_review / register_external_with_exposure](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/registry.rs#L557-L635)：规范化 ToolName 后先过 ToolPolicy；trusted 重名走 error_or_panic；external 同名跳过且记录首个冲突；默认命名空间 exec_command/shell_command 为外部保留名。

- [dispatch_any_with_state](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/registry.rs#L795-L845)：未知工具返回 RespondToModel，payload 变体不匹配返回 Fatal；该层 matches_kind 不能证明全量 JSON Schema 验证。

- [dispatch_any_with_state](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/registry.rs#L875-L926)：dispatch 先绑定环境，再 PreToolUse，再刷新已有 assessment 的权限上下文，必要时 native review，随后发 start 并执行。


## 现在应能解释什么


    能按顺序说明参数何时变成不可变动作、许可为何要绑定最终参数、去重为何必须在 handler 前发生。
    执行事实和模型可见观察是两个状态；未知副作用不能被一次异常或后置拒绝抹去。

    接回 Loop 时保留一个 Dispatcher，不让每个工具各自建立权限和重复受理规则。
    本章未实现完整 Schema、并发/持久去重、真实审批服务或 OS 隔离。
